# CardioCare — train an ECG arrhythmia classifier

Trains a CNN to classify ECG strip images into 8 arrhythmia classes, using the
MIT-BIH Arrhythmia Database from PhysioNet.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

End to end this takes roughly 1.5–3 hours on a free T4. Each step below saves
its output to Google Drive, so you can close the tab and resume.

---
> Research use only. Not a medical device.


## 1. Setup


In [ ]:
!nvidia-smi || echo 'No GPU detected — go to Runtime > Change runtime type > T4 GPU'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/cardiocare'
!mkdir -p {WORK}


In [ ]:
REPO = 'https://github.com/dilangandhi/cardiocare.git'  # <-- change this

%cd /content
![ -d cardiocare ] && (cd cardiocare && git pull) || git clone $REPO
%cd /content/cardiocare


In [ ]:
!pip install -q wfdb opencv-python-headless>=4.9 scikit-learn matplotlib tqdm
print('ready')


## 2. Build the dataset

Pulls from four PhysioNet databases:

| Database | Contributes |
|---|---|
| `mitdb` | sinus rhythms, AFib, flutter, bigeminy/trigeminy (VPB), SVT |
| `afdb` | atrial fibrillation and flutter across 23 more patients |
| `vfdb` | ventricular fibrillation |
| `cudb` | more ventricular fibrillation |

**Why four and not just MIT-BIH.** `(VFL` and `(SVTA` each appear in exactly
*one* MIT-BIH recording (207). Because records are split whole, that record
lands entirely in train or entirely in test — leaving the other split with zero
examples of the class. Drawing those rhythms from additional databases spreads
every class across enough distinct patients that a record-wise split still works.

Each database is split separately, so all four appear in train, val and test.

Expect 45–90 minutes. Run the smoke test first.


In [ ]:
# Smoke test on a few records from each database (~3 min).
# Records are given as db/record. Check the class counts before the full run.
!python ml/prepare_data.py --out /content/data_test \
    --records mitdb/207 mitdb/201 mitdb/106 afdb/04015 vfdb/418 vfdb/419 cudb/cu01 \
    --window 10 --stride 5 --augment 0


In [ ]:
# Full build across all four databases. Roughly 45-90 minutes.
# Drop --databases to use all of them; name a subset to go faster.
!python ml/prepare_data.py --out /content/data --window 10 --stride 5 --augment 1


In [ ]:
import json
meta = json.load(open('/content/data/dataset.json'))
print('split by:', meta['split_by'])

empty = []
for split in ('train','val','test'):
    counts = meta['counts'].get(split, {})
    print(f"\n{split}: {sum(counts.values())} windows")
    for k in meta['classes']:
        v = counts.get(k, 0)
        flag = '   <-- EMPTY' if v == 0 else ''
        print(f'   {k:18s} {v}{flag}')
        if v == 0: empty.append(f'{k}/{split}')

if empty:
    print('\nSTOP. These class/split combinations have no data:', empty)
    print('The model cannot learn or be scored on them. Re-run prepare_data.py')
    print('with more databases, a smaller --stride, or a different --seed.')
else:
    print('\nAll 8 classes present in all 3 splits. Good to train.')


### Look at the data before training it

If the rendered strips do not look like ECGs to you, they will not to the model either.


In [ ]:
import matplotlib.pyplot as plt, glob, random
from PIL import Image

classes = meta['classes']
fig, axes = plt.subplots(len(classes), 1, figsize=(15, 2.0*len(classes)))
for ax, cls in zip(axes, classes):
    files = glob.glob(f'/content/data/train/{cls}/*.png') or glob.glob(f'/content/data/*/{cls}/*.png')
    if not files:
        ax.text(0.5, 0.5, f'{cls}: no windows found', ha='center'); ax.axis('off'); continue
    ax.imshow(Image.open(random.choice(files)))
    ax.set_ylabel(cls, rotation=0, ha='right', va='center', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 3. Train

Transfer learning from ImageNet with class-balanced sampling and a weighted loss.

**Balanced accuracy is the metric to watch**, not plain accuracy. Ventricular
fibrillation windows are ~100× rarer than sinus rhythm, so a model that never
predicts VFib can still post a high plain accuracy while being useless.


In [ ]:
!python ml/train.py --data /content/data --arch resnet50 \
    --epochs 30 --batch-size 32 --lr 3e-4 --out /content/models


In [ ]:
# Copy checkpoints to Drive so a disconnect does not lose them.
!mkdir -p {WORK}/models && cp -v /content/models/* {WORK}/models/


### Compare architectures (optional)

Reproduces the comparison table. Roughly 30–45 minutes each.


In [ ]:
for arch in ['efficientnet_b0', 'mobilenet_v2', 'densenet121']:
    print('='*70); print(arch); print('='*70)
    !python ml/train.py --data /content/data --arch {arch} --epochs 20 --out /content/models


In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob('/content/models/*_report.json')):
    r = json.load(open(f))
    rows.append((r['arch'], r['test_accuracy'], r['test_balanced_accuracy'], r['epochs_run']))
print(f"{'architecture':20s}{'accuracy':>11}{'balanced':>11}{'epochs':>8}")
for a, acc, bal, ep in sorted(rows, key=lambda r: -r[2]):
    print(f'{a:20s}{acc:11.4f}{bal:11.4f}{ep:8d}')


## 4. Evaluate both paths

Scores the CNN **and** the untrained rule engine on the same held-out windows.
The rule engine is the baseline the CNN has to beat to justify its existence,
and the agreement rate between the two is what the interface reports to users.


In [ ]:
!python ml/evaluate.py --data /content/data \
    --checkpoint /content/models/resnet50_best.pt --out /content/figures


In [ ]:
import json
rep = json.load(open('docs/evaluation.json'))
for path in ('rules', 'cnn', 'fused'):
    if path in rep:
        r = rep[path]
        print(f"{path:8s} accuracy {r['accuracy']:.4f}   balanced {r['balanced_accuracy']:.4f}")
if 'agreement' in rep:
    print('\nagreement:', rep['agreement'])


In [ ]:
from IPython.display import Image as IPImage, display
import os
for f in ['confusion_cnn.png', 'confusion_rules.png', 'confusion_fused.png', 'roc_cnn.png']:
    p = f'/content/figures/{f}'
    if os.path.exists(p):
        print(f); display(IPImage(p))


In [ ]:
import json
rep = json.load(open('docs/evaluation.json'))
pc = rep.get('cnn', rep.get('rules', {})).get('per_class', {})
print(f"{'class':20s}{'support':>9}{'precision':>11}{'recall':>9}{'f1':>8}")
for k, v in pc.items():
    print(f"{k:20s}{v['support']:9d}{v['precision']:11.3f}{v['recall']:9.3f}{v['f1']:8.3f}")


## 5. Export for serving

ONNX rather than a torch checkpoint: onnxruntime is ~50 MB against torch's
~900 MB, which is the difference between fitting a free deployment tier and not.


In [ ]:
!python ml/export_onnx.py --checkpoint /content/models/resnet50_best.pt \
    --out /content/models/resnet50.onnx


In [ ]:
!cp -v /content/models/resnet50.onnx /content/models/resnet50.json {WORK}/models/
from google.colab import files
files.download('/content/models/resnet50.onnx')


## 6. Deploy

1. Commit `resnet50.onnx` to `models/` in your repo (use Git LFS, or attach it
   to a GitHub release and download it at container start).
2. Redeploy. The header will show the model name and weight hash, and findings
   become dual-path with corroboration.

---

### Reporting your results honestly

- Quote **balanced accuracy**, and say the split was by record.
- Report per-class recall. A high average can hide a class the model never predicts.
- State the test record IDs. They are in `dataset.json`.
- Do **not** quote the 99% figure from the synthetic self-test as accuracy — the
  generator and the rule engine share their assumptions, so it is circular.
- If your number is lower than a previously reported one, check whether the
  earlier split was window-wise. That difference is usually the whole story, and
  explaining it is a stronger result than the higher number was.
